# TxGNN Custom Split Evaluation

This notebook installs a GPU-compatible TxGNN stack in Colab, then trains and evaluates a fresh TxGNN model on your custom split with the leakage-free protocol.

Compatibility choice:
- PyTorch `2.4.1+cu124`
- DGL CUDA wheel for `torch-2.4/cu124`

Outputs:
- `split/txgnn_leakage_free_results/txgnn_leakage_free_summary.csv`
- `split/txgnn_leakage_free_results/txgnn_leakage_free_per_disease.csv`
- `split/txgnn_leakage_free_results/txgnn_leakage_free_metadata.json`
- `split/txgnn_leakage_free_results/saved_model/`


In [1]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702')
TXGNN_REPO_DIR = PROJECT_DIR / 'TxGNN'

if not PROJECT_DIR.exists():
    raise FileNotFoundError(PROJECT_DIR)
if not TXGNN_REPO_DIR.exists():
    raise FileNotFoundError(TXGNN_REPO_DIR)

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *packages]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

pip_install('--no-cache-dir', '--upgrade', 'pip', 'setuptools', 'wheel')
pip_install(
    '--no-cache-dir',
    '--force-reinstall',
    'torch==2.4.0',
    'torchvision==0.19.0',
    'torchaudio==2.4.0',
    '--index-url',
    'https://download.pytorch.org/whl/cu124',
)
pip_install('--no-deps', 'torchdata==0.8.0')
pip_install('--no-cache-dir', '-f', 'https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html', 'dgl==2.4.0+cu124')
pip_install(
    '--no-cache-dir',
    'pandas',
    'scikit-learn',
    'matplotlib',
    'tqdm',
    'goatools',
    'networkx',
    'requests',
)
pip_install('--no-cache-dir', '--force-reinstall', 'wandb==0.17.9')
pip_install('--no-cache-dir', '--force-reinstall', 'numpy==1.26.4', 'scipy==1.13.1')
pip_install('-e', str(TXGNN_REPO_DIR))

print('\nInstall finished.')
print('Now do Runtime -> Restart session, then continue with the next cells.')


Running: /usr/bin/python3 -m pip install -q --no-cache-dir --upgrade pip setuptools wheel
Running: /usr/bin/python3 -m pip install -q --no-cache-dir --force-reinstall torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124
Running: /usr/bin/python3 -m pip install -q --no-deps torchdata==0.8.0
Running: /usr/bin/python3 -m pip install -q --no-cache-dir -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html dgl==2.4.0+cu124
Running: /usr/bin/python3 -m pip install -q --no-cache-dir pandas scikit-learn matplotlib tqdm goatools networkx requests
Running: /usr/bin/python3 -m pip install -q --no-cache-dir --force-reinstall wandb==0.17.9
Running: /usr/bin/python3 -m pip install -q --no-cache-dir --force-reinstall numpy==1.26.4 scipy==1.13.1
Running: /usr/bin/python3 -m pip install -q -e /content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/TxGNN

Install finished.
Now do Runtime -> Restart session, then continue with the next cells.


In [1]:
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702')
TXGNN_REPO_DIR = PROJECT_DIR / 'TxGNN'

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
if str(TXGNN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TXGNN_REPO_DIR))

import numpy as np
import scipy
import torch

print('Project dir:', PROJECT_DIR)
print('torch:', torch.__version__)
print('numpy:', np.__version__)
print('scipy:', scipy.__version__)

if not torch.__version__.startswith('2.4.'):
    raise RuntimeError('Runtime restart required: expected torch 2.4.x after install.')
if np.__version__ != '1.26.4':
    raise RuntimeError('Runtime restart required: expected numpy 1.26.4 after install.')
if scipy.__version__ != '1.13.1':
    raise RuntimeError('Runtime restart required: expected scipy 1.13.1 after install.')

import dgl
import txgnn
import wandb

RUNTIME_DEVICE = 'cpu'
if torch.cuda.is_available():
    try:
        _probe = dgl.graph((torch.tensor([0]), torch.tensor([0]))).to('cuda:0')
        del _probe
        RUNTIME_DEVICE = 'cuda:0'
    except Exception:
        RUNTIME_DEVICE = 'cpu'

print('dgl:', dgl.__version__)
print('txgnn module:', txgnn.__file__)
print('wandb:', wandb.__version__)
print('torch.cuda.is_available():', torch.cuda.is_available())
print('runtime device:', RUNTIME_DEVICE)

if RUNTIME_DEVICE != 'cuda:0':
    raise RuntimeError('DGL is still not CUDA-enabled. Re-run the install cell in a fresh runtime and restart the session.')


Project dir: /content/drive/MyDrive/BMI Year1/BMI702/project/BMI702
torch: 2.4.0+cu124
numpy: 1.26.4
scipy: 1.13.1
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


DGL backend not selected or invalid.  Assuming PyTorch for now.


dgl: 2.4.0+cu124
txgnn module: /content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/TxGNN/txgnn/__init__.py
wandb: 0.17.9
torch.cuda.is_available(): True
runtime device: cuda:0


In [ ]:
import os

USE_WANDB = True
WANDB_PROJECT = 'TxGNN_Custom_Split'
WANDB_RUN_NAME = 'TxGNN_Leakage_Free_seed42'
LOAD_SAVED_WEIGHTS = False
SAVED_MODEL_DIR = PROJECT_DIR / 'split' / 'txgnn_leakage_free_results' / 'saved_model'

# Option 1: set USE_WANDB = True and paste your 40-character API key below.
# Option 2: export WANDB_API_KEY in Colab and leave the line below empty.
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')
WANDB_API_KEY = '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'

if LOAD_SAVED_WEIGHTS:
    print(f'Will load saved TxGNN weights from: {SAVED_MODEL_DIR}')
else:
    print('Will train TxGNN from scratch before evaluation.')

if USE_WANDB:
    import wandb
    key = WANDB_API_KEY.strip()
    if key:
        os.environ['WANDB_API_KEY'] = key
        wandb.login(key=key, relogin=True)
    else:
        wandb.login(relogin=True)
    print(f'W&B enabled: project={WANDB_PROJECT}, run={WANDB_RUN_NAME}')
else:
    print('W&B disabled. Set USE_WANDB = True to enable live train/val tracking.')


In [ ]:
import importlib
import json

import custom_txgnn_leakage_free_eval

importlib.reload(custom_txgnn_leakage_free_eval)
run_txgnn_leakage_free_eval = custom_txgnn_leakage_free_eval.run_txgnn_leakage_free_eval

summary_df, per_disease_df, metadata = run_txgnn_leakage_free_eval(
    project_dir=PROJECT_DIR,
    split_dir=PROJECT_DIR / 'split',
    output_dir=PROJECT_DIR / 'split' / 'txgnn_leakage_free_results',
    seed=42,
    valid_ratio=0.1,
    device=RUNTIME_DEVICE,
    pretrain_epochs=0,
    finetune_epochs=500,
    pretrain_lr=1e-3,
    finetune_lr=5e-4,
    batch_size=1024,
    pretrain_print_per_n=20,
    train_print_per_n=5,
    valid_per_n=20,
    balanced_repeats=100,
    use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT,
    wandb_run_name=WANDB_RUN_NAME,
    load_saved_model_dir=SAVED_MODEL_DIR if LOAD_SAVED_WEIGHTS else None,
)

display(summary_df.style.format({'TxGNN': '{:.4f}'}))
print(json.dumps(metadata, indent=2))


In [ ]:
per_disease_df.sort_values('MRR', ascending=False).head(20)


,disease_id,disease_name,candidate_drugs_after_masking,num_masked_train_valid_drugs,num_test_drugs,MRR,R@1,R@5,R@10,R@50,AUROC_per_disease,AUPRC_balanced_1to1,AUROC_balanced_1to1
61,4979.0,asthma,7908,0,38,0.038115,0.026316,0.026316,0.026316,0.236842,0.784027,0.831392,0.782645
3,10615_13006_8250_32569_9876_32567,isolated growth hormone deficiency,7908,0,1,0.023810,0.000000,0.000000,0.000000,1.000000,0.994815,1.000000,1.000000
56,33946.0,hereditary angioedema with C1Inh deficiency,7908,0,4,0.019708,0.000000,0.000000,0.000000,0.250000,0.947558,0.957833,0.955000
105,9879.0,short stature due to growth hormone qualitativ...,7908,0,1,0.015625,0.000000,0.000000,0.000000,0.000000,0.992032,1.000000,1.000000
103,9830.0,parkinsonian-pyramidal syndrome,7908,0,22,0.015312,0.000000,0.000000,0.045455,0.227273,0.934764,0.933381,0.934814
13,11764_11658_13625_8199_14604_11613_14233_11562...,Parkinson disease,7908,0,27,0.014698,0.000000,0.000000,0.037037,0.185185,0.926993,0.926995,0.926269
48,19614.0,pituitary deficiency due to Rathke's pouch cysts,7908,0,1,0.013158,0.000000,0.000000,0.000000,0.000000,0.990515,0.995000,0.990000
2,10604_10602_15720_15719_18660_15721_15715_1571...,hemophilia,7908,0,14,0.011586,0.000000,0.000000,0.000000,0.000000,0.989214,0.975299,0.987041
95,9100.0,IDDM 1,7908,0,4,0.011261,0.000000,0.000000,0.000000,0.000000,0.988866,0.981500,0.985000
63,5100.0,systemic sclerosis,7908,0,3,0.010979,0.000000,0.000000,0.000000,0.333333,0.840481,0.899667,0.848889


## Pretrain


In [5]:
import importlib
import custom_txgnn_pretrained_eval
importlib.reload(custom_txgnn_pretrained_eval)

summary_df, per_disease_df, metadata = custom_txgnn_pretrained_eval.run_txgnn_pretrained_eval(
    project_dir=PROJECT_DIR,
    split_dir=PROJECT_DIR / 'split',
    output_dir=PROJECT_DIR / 'split' / 'txgnn_leakage_free_pretrained_results',
    device=RUNTIME_DEVICE,
    pretrain_epochs=2,
    finetune_epochs=500,
    use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT,
    wandb_run_name='TxGNN_Leakage_Free_Pretrained_seed42',
)

In [ ]:
display(summary_df.style.format({'TxGNN': '{:.4f}'}))
print(json.dumps(metadata, indent=2))
per_disease_df.sort_values('MRR', ascending=False).head(20)


## Neighborhood-Intact Custom Split

This experiment keeps test diseases' phenotype and other neighborhood edges intact during training, but removes held-out `indication` and `contraindication` edges for validation/test diseases from the training graph.


In [ ]:
import os

USE_WANDB = True
WANDB_PROJECT = 'TxGNN_Custom_Split'
WANDB_RUN_NAME = 'TxGNN_Neighborhood_Intact_seed42'
LOAD_SAVED_WEIGHTS = False
SAVED_MODEL_DIR = PROJECT_DIR / 'split' / 'txgnn_neighborhood_intact_results' / 'saved_model'

# Option 1: set USE_WANDB = True and paste your 40-character API key below.
# Option 2: export WANDB_API_KEY in Colab and leave the line below empty.
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')
WANDB_API_KEY = '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'

if LOAD_SAVED_WEIGHTS:
    print(f'Will load saved TxGNN weights from: {SAVED_MODEL_DIR}')
else:
    print('Will train TxGNN from scratch before evaluation.')

if USE_WANDB:
    import wandb
    key = WANDB_API_KEY.strip()
    if key:
        os.environ['WANDB_API_KEY'] = key
        wandb.login(key=key, relogin=True)
    else:
        wandb.login(relogin=True)
    print(f'W&B enabled: project={WANDB_PROJECT}, run={WANDB_RUN_NAME}')
else:
    print('W&B disabled. Set USE_WANDB = True to enable live train/val tracking.')


In [5]:
import importlib
import json

import custom_txgnn_neighborhood_intact_eval

importlib.reload(custom_txgnn_neighborhood_intact_eval)
run_txgnn_neighborhood_intact_eval = custom_txgnn_neighborhood_intact_eval.run_txgnn_neighborhood_intact_eval

summary_df, per_disease_df, metadata = run_txgnn_neighborhood_intact_eval(
    project_dir=PROJECT_DIR,
    split_dir=PROJECT_DIR / 'split',
    output_dir=PROJECT_DIR / 'split' / 'txgnn_neighborhood_intact_results',
    seed=42,
    valid_ratio=0.1,
    device=RUNTIME_DEVICE,
    pretrain_epochs=2,
    finetune_epochs=500,
    pretrain_lr=1e-3,
    finetune_lr=5e-4,
    batch_size=1024,
    pretrain_print_per_n=20,
    train_print_per_n=5,
    valid_per_n=20,
    balanced_repeats=100,
    use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT,
    wandb_run_name=WANDB_RUN_NAME,
    load_saved_model_dir=SAVED_MODEL_DIR if LOAD_SAVED_WEIGHTS else None,
)

display(summary_df.style.format({'TxGNN': '{:.4f}'}))
print(json.dumps(metadata, indent=2))


Preparing neighborhood-intact dataframes and graphs for TxGNN...
Initializing TxGNN model...


wandb: Currently logged in as: beatrice-chen (bmi702). Use `wandb login --relogin` to force relogin


Skipping TxGNN pretraining because this DGL build does not expose dgl.dataloading.EdgeDataLoader. Proceeding directly to finetuning.
Starting TxGNN finetuning...
Epoch: 0 LR: 0.00050 Loss 0.6930, Train Micro AUROC 0.5133 Train Micro AUPRC 0.5125 Train Macro AUROC 0.5137 Train Macro AUPRC 0.5060
----- AUROC Performance in Each Relation -----
('drug', 'contraindication', 'disease'): 0.5122748136409097
('drug', 'indication', 'disease'): 0.4821690288968498
('drug', 'off-label use', 'disease'): 0.5082967580865869
('disease', 'rev_contraindication', 'drug'): 0.5260189392563657
('disease', 'rev_indication', 'drug'): 0.5794152537163799
('disease', 'rev_off-label use', 'drug'): 0.47392588265835933
----- AUPRC Performance in Each Relation -----
('drug', 'contraindication', 'disease'): 0.5065218950084597
('drug', 'indication', 'disease'): 0.4941691717522702
('drug', 'off-label use', 'disease'): 0.5072501349696825
('disease', 'rev_contraindication', 'drug'): 0.5210218932283448
('disease', 'rev_ind

  0%|          | 0/108 [00:00<?, ?it/s]

,Metric,TxGNN
0,MRR,0.0131
1,R@1,0.0003
2,R@5,0.0122
3,R@10,0.0380
4,R@50,0.1552
5,AUROC (per-disease mean),0.6062
6,AUPRC (balanced 1:1),0.7537
7,AUROC (balanced 1:1),0.6048


{
  "seed": 42,
  "device": "cuda:0",
  "requested_device": "cuda:0",
  "valid_ratio": 0.1,
  "pretrain_epochs": 2,
  "finetune_epochs": 500,
  "pretrain_lr": 0.001,
  "finetune_lr": 0.0005,
  "batch_size": 1024,
  "balanced_repeats": 100,
  "use_wandb": true,
  "wandb_project": "TxGNN_Custom_Split",
  "wandb_run_name": "TxGNN_Neighborhood_Intact_seed42",
  "loaded_saved_model_dir": null,
  "results_dir": "/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/split/txgnn_neighborhood_intact_results",
  "summary_path": "/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/split/txgnn_neighborhood_intact_results/txgnn_neighborhood_intact_summary.csv",
  "per_disease_path": "/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/split/txgnn_neighborhood_intact_results/txgnn_neighborhood_intact_per_disease.csv",
  "raw_eval_path": "/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/split/txgnn_neighborhood_intact_results/txgnn_neighborhood_intact_raw_eval.pkl",
  "saved_model_dir": 

In [ ]:
per_disease_df.sort_values('MRR', ascending=False).head(20)


,disease_id,disease_name,candidate_drugs_after_masking,num_masked_train_valid_drugs,num_test_drugs,MRR,R@1,R@5,R@10,R@50,AUROC_per_disease,AUPRC_balanced_1to1,AUROC_balanced_1to1
3,10615_13006_8250_32569_9876_32567,isolated growth hormone deficiency,7957,0,1,0.125000,0.000000,0.000000,1.000000,1.000000,0.999120,1.000000,1.000000
88,8170.0,ovarian cancer,7957,0,15,0.113943,0.000000,0.200000,0.333333,1.000000,0.998413,0.996570,0.997867
6,10917_7319_1314,chondrocalcinosis,7957,0,8,0.088545,0.000000,0.000000,0.500000,1.000000,0.998490,0.997987,0.998281
19,1266.0,erysipelas,7957,0,3,0.065829,0.000000,0.000000,0.333333,0.666667,0.994468,0.996667,0.995556
99,9348.0,classic Hodgkin lymphoma,7957,0,21,0.058600,0.000000,0.095238,0.190476,0.619048,0.983229,0.983849,0.983379
82,7739.0,Huntington disease,7957,0,2,0.050518,0.000000,0.000000,0.500000,0.500000,0.938906,0.955833,0.932500
85,7794_9239_9223_13912_13961_7844_13946_14105_13...,hypogonadotropic hypogonadism with or without ...,7957,0,2,0.050109,0.000000,0.000000,0.500000,0.500000,0.710182,0.827500,0.695000
77,7126_8468_24512_13192,"spondyloarthropathy, susceptibility to",7957,0,23,0.047471,0.000000,0.086957,0.217391,0.347826,0.824333,0.869982,0.829282
58,4114.0,urinary bladder small cell neuroendocrine carc...,7957,0,2,0.041461,0.000000,0.000000,0.000000,0.500000,0.993840,1.000000,1.000000
101,9417_18555,hypogonadotropic hypogonadism,7957,0,2,0.035817,0.000000,0.000000,0.000000,0.500000,0.692458,0.825833,0.685000
